In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import DataLoader, Dataset
import csv

# 데이터 로드 및 전처리
bg_rating = pd.read_csv('C:/Users/dipreez/Desktop/졸작/Board-game-recommendation/project/archive/bgg-15m-reviews.csv', on_bad_lines='skip',low_memory=False)
bg_rating.drop('comment', axis=1, inplace=True)
bg_rating = bg_rating[['user', 'name', 'rating']].dropna()

In [3]:
torch.cuda.is_available()

True

In [4]:
from sklearn.preprocessing import MinMaxScaler
# 사용자 ID와 아이템 ID를 인덱스로 변환
user_ids = bg_rating['user'].astype('category').cat.codes
item_ids = bg_rating['name'].astype('category').cat.codes
bg_rating['rating'] = pd.to_numeric(bg_rating['rating'], errors='coerce')
scaler = MinMaxScaler()
ratings = scaler.fit_transform(bg_rating['rating'].values.reshape(-1, 1)).flatten().astype('float32')


# 훈련 데이터셋과 테스트 데이터셋 분리
train_data, test_data, train_labels, test_labels = train_test_split(
    np.vstack((user_ids, item_ids)).T, ratings, test_size=0.2, random_state=42)

In [5]:
class RatingDataset(Dataset):
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        user = self.data[idx, 0]
        item = self.data[idx, 1]
        rating = self.labels[idx]

        # Convert each to a tensor
        user_tensor = torch.tensor(user, dtype=torch.long)
        item_tensor = torch.tensor(item, dtype=torch.long)
        rating_tensor = torch.tensor(rating, dtype=torch.float32)

        input_tensor = torch.stack([user_tensor, item_tensor], dim=-1)

        return input_tensor, rating_tensor

train_dataset = RatingDataset(train_data, train_labels)
test_dataset = RatingDataset(test_data, test_labels)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [6]:
import torch.nn as nn
import torch.nn.functional as F

class NCF(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim, hidden_layers):
        super(NCF, self).__init__()
        # GMF part
        self.user_embedding_gmf = nn.Embedding(num_users, embedding_dim)
        self.item_embedding_gmf = nn.Embedding(num_items, embedding_dim)

        # MLP part
        self.user_embedding_mlp = nn.Embedding(num_users, embedding_dim)
        self.item_embedding_mlp = nn.Embedding(num_items, embedding_dim)

        self.mlp_layers = nn.Sequential()
        input_dim = 2 * embedding_dim

        self.mlp_layers.add_module("linear_0", nn.Linear(input_dim, hidden_layers[0]))
        self.mlp_layers.add_module("relu_0", nn.ReLU())

        for i, hidden_dim in enumerate(hidden_layers[1:], start=1):
            self.mlp_layers.add_module(f"linear_{i}", nn.Linear(hidden_layers[i-1], hidden_dim))
            self.mlp_layers.add_module(f"relu_{i}", nn.ReLU())

        # Combine GMF and MLP
        self.final_linear = nn.Linear(hidden_layers[-1] + embedding_dim, 1)

    def forward(self, x):
        user_id = x[:, 0]
        item_id = x[:, 1]

        # GMF part
        gmf_user_embedding = self.user_embedding_gmf(user_id)
        gmf_item_embedding = self.item_embedding_gmf(item_id)
        gmf_output = gmf_user_embedding * gmf_item_embedding

        # MLP part
        mlp_user_embedding = self.user_embedding_mlp(user_id)
        mlp_item_embedding = self.item_embedding_mlp(item_id)
        mlp_input = torch.cat([mlp_user_embedding, mlp_item_embedding], dim=-1)
        mlp_output = self.mlp_layers(mlp_input)

        # Combine GMF and MLP outputs
        concat_output = torch.cat([gmf_output, mlp_output], dim=-1)
        prediction = torch.sigmoid(self.final_linear(concat_output))

        return prediction.squeeze()

# 모델 초기화
num_users = len(bg_rating['user'].unique())
num_items = len(bg_rating['name'].unique())
embedding_dim = 10
hidden_layers = [32, 16]

model = NCF(num_users, num_items, embedding_dim, hidden_layers).to("cuda:0")

In [7]:
print(model)

NCF(
  (user_embedding_gmf): Embedding(351048, 10)
  (item_embedding_gmf): Embedding(18984, 10)
  (user_embedding_mlp): Embedding(351048, 10)
  (item_embedding_mlp): Embedding(18984, 10)
  (mlp_layers): Sequential(
    (linear_0): Linear(in_features=20, out_features=32, bias=True)
    (relu_0): ReLU()
    (linear_1): Linear(in_features=32, out_features=16, bias=True)
    (relu_1): ReLU()
  )
  (final_linear): Linear(in_features=26, out_features=1, bias=True)
)


In [11]:
from tqdm import tqdm
import torch.optim as optim
import numpy as np

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

patience = 3
best_loss = np.inf
patience_counter = 0

num_epochs = 50

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for inputs, labels in tqdm(train_loader):
        inputs = inputs.to("cuda:0")
        labels = labels.to("cuda:0")

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels.squeeze())
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)

    model.eval()

    running_loss = 0.0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to("cuda:0")
            labels = labels.to("cuda:0")
    
            outputs = model(inputs)
            loss = criterion(outputs, labels.squeeze())
            running_loss += loss.item()

    valid_loss = running_loss / len(test_loader)
    print(f"Epoch {epoch+1} Train Loss: {train_loss:.4f} Test Loss: {valid_loss:.4f}")

    if best_loss > valid_loss:
        torch.save(model.state_dict(), "best_model.pth")
        best_loss = valid_loss
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"No improvement. Patience: {patience_counter}/{patience}")
        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

100%|█████████████████████████████████████████████████████████████████████████| 197791/197791 [24:13<00:00, 136.11it/s]


Epoch 1 Train Loss: 0.0155 Test Loss: 0.0159


100%|█████████████████████████████████████████████████████████████████████████| 197791/197791 [22:28<00:00, 146.71it/s]


Epoch 2 Train Loss: 0.0153 Test Loss: 0.0159
No improvement. Patience: 1/3


100%|█████████████████████████████████████████████████████████████████████████| 197791/197791 [24:27<00:00, 134.83it/s]


...

Epoch 6 Train Loss: 0.0139 Test Loss: 0.0158
No improvement. Patience: 1/3


100%|█████████████████████████████████████████████████████████████████████████| 197791/197791 [16:49<00:00, 195.96it/s]


Epoch 7 Train Loss: 0.0136 Test Loss: 0.0158
No improvement. Patience: 2/3


100%|█████████████████████████████████████████████████████████████████████████| 197791/197791 [16:01<00:00, 205.73it/s]


Epoch 8 Train Loss: 0.0133 Test Loss: 0.0158
No improvement. Patience: 3/3
Early stopping triggered.


In [ ]:
# 테스트 평가
model.eval()
with torch.no_grad():
    mse = 0.0
    for inputs, labels in test_loader:
        outputs = model(inputs)
        mse += criterion(outputs, labels.squeeze()).item()

    mse /= len(test_loader)
    print(f"Test MSE: {mse}")

In [ ]:
# 모델 저장
    torch.save(model.state_dict(), 'ncf_model.pth')